In [2]:
import numpy as np
import pandas as pd
import time
from utils import TECHNICAL_INDICATORS, MACRO, CUSTOM_NUMERICAL, CATEGORICAL

In [3]:
df_full = pd.read_parquet('/Users/vancescadinh/Documents/zoomcamp/stock-markets-analysis/homework3/stocks_df_combined_2025_06_13.parquet')

In [4]:
df = df_full[df_full.Date>='2000-01-01']
# dummy variables are not generated from Date and numeric variables
df.loc[:,'Month'] = df.Month.dt.strftime('%B')
df.loc[:,'Weekday'] = df.Weekday.astype(str)

/var/folders/hq/_jjt9xg57mlc66c7s8mm5d1c0000gn/T/ipykernel_10838/4197779462.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['January' 'January' 'January' ... 'June' 'June' 'June']' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  df.loc[:,'Month'] = df.Month.dt.strftime('%B')
/var/folders/hq/_jjt9xg57mlc66c7s8mm5d1c0000gn/T/ipykernel_10838/4197779462.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['0' '1' '2' ... '2' '3' '4']' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.loc[:,'Weekday'] = df.Weekday.astype(str)


## Question 1: Dummies for Month and Week-of-Month
What is the ABSOLUTE CORRELATION VALUE of the most correlated dummy variable _w<week_of_month> with the binary outcome is_positive_growth_30d_future?

From the correlation analysis and modeling, you may have observed that October and November are potentially important seasonal months. In this task, you'll go further by generating dummy variables for both the Month and Week-of-Month (starting from 1). For example, the first week of October should be coded as: 'October_w1'.

Once you've generated these new variables, identify the one with the highest absolute correlation with is_positive_growth_30d_future, and round the result to three decimal places.

In [4]:
GROWTH = [g for g in df_full.keys() if (g.find('growth_')==0)&(g.find('future')<0)]
TO_PREDICT = [g for g in df_full.keys() if (g.find('future')>=0)]
TECHNICAL_PATTERNS = [g for g in df_full.keys() if g.find('cdl')>=0]

NUMERICAL = GROWTH + TECHNICAL_INDICATORS + TECHNICAL_PATTERNS + CUSTOM_NUMERICAL + MACRO

In [ ]:
# Create dummy variables
dummy_variables = pd.get_dummies(df[CATEGORICAL], dtype='int32')
df.loc[:,'wom'] = ((df['Date'].dt.day - 1) // 7 + 1).astype(int)
df.loc[:, 'month_wom'] = df['Month'] + '_week' + df['wom'].astype(str)

CATEGORICAL.append('month_wom')

dummies = pd.get_dummies(df[CATEGORICAL])
DUMMIES = dummies.keys().to_list()

In [6]:
# Concatenate the dummy variables with the original DataFrame
df_with_wom_dummies = pd.concat([df, dummies], axis=1)

In [7]:
corr_is_positive_growth_30d_future_df = pd.DataFrame(df_with_wom_dummies[NUMERICAL+DUMMIES+TO_PREDICT].corr()['is_positive_growth_30d_future']).T
dv_month_wom = [w for w in corr_is_positive_growth_30d_future_df.keys() if w.find('month_wom')>=0]
dv_month_wom_df = corr_is_positive_growth_30d_future_df[dv_month_wom].T
dv_month_wom_df['abs_corr'] = dv_month_wom_df['is_positive_growth_30d_future'].abs().round(3)
dv_month_wom_df['abs_corr'].sort_values(ascending=False)

month_wom_October_week4      0.025
month_wom_November_week3     0.022
month_wom_November_week2     0.019
month_wom_October_week3      0.018
month_wom_January_week2      0.018
month_wom_February_week1     0.017
month_wom_January_week5      0.017
month_wom_January_week3      0.017
month_wom_January_week4      0.015
month_wom_September_week4    0.014
month_wom_November_week4     0.013
month_wom_March_week2        0.013
month_wom_October_week1      0.013
month_wom_October_week5      0.013
month_wom_February_week2     0.013
month_wom_September_week5    0.013
month_wom_March_week4        0.012
month_wom_August_week4       0.012
month_wom_May_week1          0.011
month_wom_December_week5     0.010
month_wom_July_week4         0.010
month_wom_November_week1     0.010
month_wom_March_week3        0.010
month_wom_August_week3       0.010
month_wom_February_week3     0.010
month_wom_June_week3         0.010
month_wom_June_week5         0.009
month_wom_June_week4         0.009
month_wom_September_

## Question 2: Define New "Hand" Rules on Macro and Technical Indicator Variables
What is the precision score for the best of the NEW predictions (pred3 or pred4), rounded to 3 digits after the comma?

In [5]:
from utils import temporal_split

In [7]:
dummy_variables = pd.get_dummies(df[CATEGORICAL], dtype='int32')
df_with_dummies = pd.concat([df, dummy_variables], axis=1)

In [8]:
min_date_df = df_with_dummies.Date.min()
max_date_df = df_with_dummies.Date.max()

df_with_dummies = temporal_split(df_with_dummies,
                                 min_date = min_date_df,
                                 max_date = max_date_df)

# remove the "segmentation" problem (warning message on df performance after many joins and data transformations)
new_df = df_with_dummies.copy()

In [9]:
# generate manual predictions
# Let's label all prediction features with prefix "pred"
new_df['pred0_manual_cci'] = (new_df.cci>200).astype(int)
new_df['pred1_manual_prev_g1'] = (new_df.growth_30d>1).astype(int)
new_df['pred2_manual_prev_g1_and_snp'] = ((new_df['growth_30d'] > 1) & (new_df['growth_snp500_30d'] > 1)).astype(int)
new_df['pred3_manual_dgs10_5'] = ((new_df['DGS10'] <=4) & (new_df['DGS5'] <=1)).astype(int)
new_df['pred4_manual_dgs10_fedfunds'] = ((new_df['DGS10'] > 4) & (new_df['FEDFUNDS']<= 4.795)).astype(int)

In [10]:
PREDICTIONS = [k for k in new_df.keys() if k.startswith('pred')]
PREDICTIONS

['pred0_manual_cci',
 'pred1_manual_prev_g1',
 'pred2_manual_prev_g1_and_snp',
 'pred3_manual_dgs10_5',
 'pred4_manual_dgs10_fedfunds']

In [12]:
# generate columns is_correct_
for pred in PREDICTIONS:
  part1 = pred.split('_')[0] # first prefix before '_'
  new_df[f'is_correct_{part1}'] =  (new_df[pred] == new_df.is_positive_growth_30d_future).astype(int)

In [13]:
new_df.head(2)

,Open,High,Low,Close_x,Volume,Dividends,Stock Splits,Ticker,Year,Month,Weekday,Date,growth_1d,growth_3d,growth_7d,growth_30d,growth_90d,growth_365d,growth_future_30d,SMA10,SMA20,growing_moving_average,high_minus_low_relative,volatility,is_positive_growth_30d_future,ticker_type,index_x,adx,adxr,apo,aroon_1,aroon_2,aroonosc,bop,cci,cmo,dx,macd,macdsignal,macdhist,...,Ticker_CDI.PA,Ticker_GOOG,Ticker_HDB,Ticker_HINDUNILVR.NS,Ticker_IBN,Ticker_IDEXY,Ticker_INFY,Ticker_ITC.NS,Ticker_JPM,Ticker_LICI.NS,Ticker_LLY,Ticker_LT.NS,Ticker_MC.PA,Ticker_META,Ticker_MSFT,Ticker_NVDA,Ticker_NVO,Ticker_OR.PA,Ticker_RELIANCE.NS,Ticker_RMS.PA,Ticker_SAP,Ticker_SBIN.NS,Ticker_SIE.DE,Ticker_TCS.NS,Ticker_TTE,Ticker_V,ticker_type_EU,ticker_type_INDIA,ticker_type_US,split,pred0_manual_cci,pred1_manual_prev_g1,pred2_manual_prev_g1_and_snp,pred3_manual_dgs10_5,pred4_manual_dgs10_fedfunds,is_correct_pred0,is_correct_pred1,is_correct_pred2,is_correct_pred3,is_correct_pred4
3490,35.975752,36.358881,34.328300,35.726719,53228400.0,0.0,0.0,MSFT,2000,January,0,2000-01-03,0.998394,0.988341,0.991494,1.372333,1.222950,2.063054,0.845576,35.833990,33.234468,1,0.056836,0.385108,0,US,3490,38.756284,27.379214,3.584027,0.0,85.714286,85.714286,-0.122642,26.847443,43.796220,26.148203,2.018063,1.950544,0.067519,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,train,0,1,1,0,0,1,0,0,1,1
3491,34.807221,35.899137,34.404936,34.519875,54119000.0,0.0,0.0,MSFT,2000,January,1,2000-01-04,0.966220,0.957493,0.959021,1.309593,1.190225,1.979132,0.866815,35.830161,33.497869,1,0.043285,0.406198,0,US,3491,37.855707,28.288136,3.303435,0.0,78.571429,78.571429,-0.192308,-34.318580,21.380083,26.148203,1.830729,1.926581,-0.095852,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,train,0,1,0,0,0,1,0,1,1,1


In [17]:
IS_CORRECT =  [k for k in new_df.keys() if k.startswith('is_correct_')]

for i, is_correct_column in enumerate(IS_CORRECT):
    prediction_column = PREDICTIONS[i]
    filter = (new_df.split == 'test') & (new_df[prediction_column] == 1)

    # Subset where model predicted positive (1)
    predicted_positives = new_df[filter]

    # Count true positives (correct predictions among predicted positives)
    true_positives = predicted_positives[is_correct_column].sum()

    # Precision: TP / (TP + FP) = TP / total predicted positives
    total_predicted_positives = len(predicted_positives)
    precision = true_positives / total_predicted_positives
    precision = precision.round(3)
    print(f'Prediction column: {prediction_column}, is_correct_column: {is_correct_column}')
    print(f'Precision: {precision:.4f} ({true_positives}/{total_predicted_positives})')
    print('---------')

Prediction column: pred0_manual_cci, is_correct_column: is_correct_pred0
Precision: 0.5580 (443/794)
---------
Prediction column: pred1_manual_prev_g1, is_correct_column: is_correct_pred1
Precision: 0.5420 (9748/17991)
---------
Prediction column: pred2_manual_prev_g1_and_snp, is_correct_column: is_correct_pred2
Precision: 0.5220 (6984/13367)
---------
Prediction column: pred3_manual_dgs10_5, is_correct_column: is_correct_pred3
Precision: 0.5800 (578/997)
---------
Prediction column: pred4_manual_dgs10_fedfunds, is_correct_column: is_correct_pred4
Precision: 0.4660 (2640/5660)
---------


## Question 3: Unique Correct Predictions from a 10-Level Decision Tree Classifier (pred5_clf_10)
What is the total number of records in the TEST dataset where the new prediction pred5_clf_10 is correct, while all 'hand' rule predictions (pred0 to pred4) are incorrect?

In [6]:
from utils import remove_infinite_values, fit_decision_tree, predict_decision_tree, temporal_split

In [7]:
dummy_variables = pd.get_dummies(df[CATEGORICAL], dtype='int32')
df_with_dummies = pd.concat([df, dummy_variables], axis=1)

min_date_df = df_with_dummies.Date.min()
max_date_df = df_with_dummies.Date.max()

df_with_dummies = temporal_split(df_with_dummies,
                                 min_date = min_date_df,
                                 max_date = max_date_df)

# remove the "segmentation" problem (warning message on df performance after many joins and data transformations)
new_df = df_with_dummies.copy()

In [8]:
GROWTH = [g for g in df_full.keys() if (g.find('growth_')==0)&(g.find('future')<0)]
TECHNICAL_PATTERNS = [g for g in df_full.keys() if g.find('cdl')>=0]
NUMERICAL = GROWTH + TECHNICAL_INDICATORS + TECHNICAL_PATTERNS + CUSTOM_NUMERICAL + MACRO
dummies = pd.get_dummies(df[CATEGORICAL])
DUMMIES = dummies.keys().to_list()

# Split the data into training and testing sets based on the split date
features_list = NUMERICAL+DUMMIES
to_predict = 'is_positive_growth_30d_future'

train_df = new_df[new_df.split.isin(['train','validation'])].copy(deep=True)
test_df = new_df[new_df.split.isin(['test'])].copy(deep=True)

# ONLY numerical Separate features and target variable for training and testing sets
# need Date and Ticker later when merging predictions to the dataset
X_train = train_df[features_list+[to_predict,'Date','Ticker']]
X_test = test_df[features_list+[to_predict,'Date','Ticker']]

print(f'length: X_train {X_train.shape},  X_test {X_test.shape}')

length: X_train (160387, 241),  X_test (31408, 241)


In [9]:
# Can't have +-inf values . E.g. ln(volume)=-inf when volume==0 => substitute with 0

# Disable SettingWithCopyWarning
pd.options.mode.chained_assignment = None  # default='warn'

X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)

# Need to fill NaNs somehow
X_train.fillna(0, inplace=True)
X_test.fillna(0, inplace=True)

print(f'length: X_train_imputed {X_train.shape},  X_test_imputed {X_test.shape}')

length: X_train_imputed (160387, 241),  X_test_imputed (31408, 241)


In [10]:
X_train_imputed = X_train # we won't use outliers removal to save more data to train: remove_outliers_percentile(X_train)
X_test_imputed = X_test # we won't use outliers removal to save more data to test: remove_outliers_percentile(X_test)

In [11]:
y_train = X_train_imputed[to_predict]
y_test = X_test_imputed[to_predict]

# remove y_train, y_test from X_ dataframes
del X_train_imputed[to_predict]
del X_test_imputed[to_predict]

## Question 4: Hyperparameter tuning for a Decision Tree
What is the optimal tree depth (from 1 to 20) for a DecisionTreeClassifier?

In [12]:
%%time
# drop 2 columns before fitting the tree, but we need those columns later for joins
clf_20, train_columns = fit_decision_tree(X=X_train_imputed.drop(['Date','Ticker'],axis=1),
                           y=y_train,
                           max_depth=20)

CPU times: user 20.5 s, sys: 419 ms, total: 21 s
Wall time: 21.4 s


In [13]:
%%time
clf_10, train_columns = fit_decision_tree(X=X_train_imputed.drop(['Date','Ticker'],axis=1),
                           y=y_train,
                           max_depth=10)

CPU times: user 12.5 s, sys: 333 ms, total: 12.8 s
Wall time: 13.1 s


In [14]:
%%time
clf_5, train_columns = fit_decision_tree(X=X_train_imputed.drop(['Date','Ticker'],axis=1),
                           y=y_train,
                           max_depth=5)

CPU times: user 6.98 s, sys: 355 ms, total: 7.34 s
Wall time: 7.69 s


In [15]:
%%time
clf_15, train_columns = fit_decision_tree(X=X_train_imputed.drop(['Date','Ticker'],axis=1),
                           y=y_train,
                           max_depth=15)

CPU times: user 17 s, sys: 313 ms, total: 17.3 s
Wall time: 17.6 s


In [16]:
pred20 = predict_decision_tree(clf_20, X_test_imputed.drop(['Date','Ticker'],axis=1), y_test)
# Predictions of a decision tree of depth "20"
# pred20.pred_.value_counts()

Maximum depth of the decision tree: 20
Accuracy =0.5419001528273052, precision = 0.5813733140118158


In [17]:
pred10 = predict_decision_tree(clf_10, X_test_imputed.drop(['Date','Ticker'],axis=1), y_test)
# Predictions of a decision tree of depth "10"
pred10.pred_.value_counts()

Maximum depth of the decision tree: 10
Accuracy =0.5289416709118696, precision = 0.5613196037284661


pred_
1    20491
0    10917
Name: count, dtype: int64

In [18]:
pred15 = predict_decision_tree(clf_15, X_test_imputed.drop(['Date','Ticker'],axis=1), y_test)
# Predictions of a decision tree of depth "15"
pred15.pred_.value_counts()

Maximum depth of the decision tree: 15
Accuracy =0.5413907284768212, precision = 0.5793963254593176


pred_
1    18288
0    13120
Name: count, dtype: int64

In [19]:
pred5 = predict_decision_tree(clf_5, X_test_imputed.drop(['Date','Ticker'],axis=1), y_test)
# Predictions of a decision tree of depth "5"
pred5.pred_.value_counts()

Maximum depth of the decision tree: 5
Accuracy =0.5991467142129394, precision = 0.627845220030349


pred_
1    18452
0    12956
Name: count, dtype: int64